# Imports

In [1]:
import pyspark
from typing import List, Union, Dict
import pyspark.sql.functions as f
from dateutil.relativedelta import relativedelta
from pyspark.sql.window import Window
import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from pyspark import SparkConf
from pyspark.sql import SparkSession


/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
parameters = {
    "spark.driver.maxResultSize": "50g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow -Dio.netty.tryReflectionSetAccessible=true",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow -Dio.netty.tryReflectionSetAccessible=true",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.sql.legacy.timeParserPolicy": "LEGACY",
    "spark.driver.memory": "50g",
}

spark_conf = SparkConf().setAll(parameters.items())

In [3]:
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/02 11:40:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load data

In [4]:
int_transactions = spark.read.parquet("../data/02_intermediate/int_transactions")
prm_spine = spark.read.parquet("../data/03_primary/prm_spine")

In [110]:
categories_to_drop = []
categories_to_keep = []
sectors_to_drop = []
sectors_to_drop_additional = []
products_to_drop = []


initial_date = initial_date_nptb_pull = '2023-01-01'
n_quarter_nptb_pull = 4
n_recommendations_pull = 10
min_number_of_trx_for_recommendations = 3

n_quarter_nptb_push = 6
last_date = "2025-01-01" # First date of predictions
minimum_orders_to_be_considered = 2
products_to_drop_push = [
    "SERVICE CHARGES 10 SAR | SERVICE | SERVICE"
]

als_params = {
    "factors": 3,
    "regularization": 0.08,
    "alpha": 0.17,
    "random_state": 123,
}
recommendations_per_user = 10

# Pull

## Clean transactions

In [ ]:
sectors_to_drop = list(set(sectors_to_drop).union(set(sectors_to_drop_additional)))
relevant_ids = master.filter(
    ~master.sector_id_original.isin(sectors_to_drop)
).select(
    f.col('_id').alias('client_id')
).dropDuplicates()
transactions = transactions.join(relevant_ids, on='client_id', how='inner')
transactions = transactions.filter(
    ~(
    ((f.col('transaction_typ')=='FA') & (f.col('total_product_val')<=0))|
    (f.col('total_catalog_product_val')<f.col('total_product_val'))
    )
)

products = products.withColumn(
    "category_name",
    f.regexp_replace(f.lower(f.col("category_name")), ' ', '_')
).drop('product_name')

products = products.filter(~f.col('product_id').isin(products_to_drop))

transactions = transactions.join(
    products,
    on=["product_id"],
    how="left"
)
categories_to_drop = list(set(categories_to_drop)-set(categories_to_keep))
transactions = transactions.withColumn(
    'category_name',
    f.when(
        (f.col('category_name')=='premios') & 
        ((f.col('total_product_val')>1) | (f.col('total_catalog_product_val')>1)),
        f.lit('others')
    ).otherwise(f.col('category_name'))
)
transactions = transactions.filter(~f.col('category_name').isin(categories_to_drop))
transactions = transactions.withColumn('year_month', f.col('transaction_dt').substr(0,7))

## Prepare transactions

In [ ]:
initial_date_n_months_before = (pd.to_datetime(initial_date) + pd.offsets.MonthEnd(0)) - relativedelta(months=n_months_nptb_pull)

transactions = transactions.withColumn(
    'transaction_month', 
    f.last_day('transaction_dt')
).filter(
    f.col('transaction_month') >= initial_date_n_months_before
).select(
    'transaction_month',
    'client_id',
    'product_id',
    'product_name',
    'product_qty',
    'total_product_val',
    'total_catalog_product_val'
).groupBy(
    ['transaction_month', 'client_id', 'product_id', 'product_name']
).agg(
    f.sum('product_qty').alias('product_qty'),
    f.sum('total_product_val').alias('total_product_val'),
    f.sum('total_catalog_product_val').alias('total_catalog_product_val')
)

out = transactions.orderBy(['client_id', 'product_id','transaction_month'])

In [ ]:
initial_date_n_months_before = pd.to_datetime(initial_date) - relativedelta(months=n_quarter_nptb_pull * 3)

In [99]:
int_transactions = int_transactions.withColumn(
    "_id", 
    f.concat_ws("__", *["customer_id", "customer_vehicle_id"])
).withColumn(
    "_observ_end_dt",
    f.to_date(f.date_trunc("quarter", f.col("transaction_dt")))
)

base_transactions = prm_spine.join(
    int_transactions,
    on=["_id", "_observ_end_dt"],
    how="inner"
)

In [ ]:
transactions = base_transactions.filter(
    f.col("_observ_end_dt") >= initial_date_n_months_before
).select(
    "_observ_end_dt",
    "_id",
    "customer_id",
    "customer_vehicle_id",
    "product_category",
    "product_id",
    "quantity",
    "sales_amount_net",
    "total_profit",
).groupBy(
    "_observ_end_dt", "_id", "customer_id", "customer_vehicle_id", "product_category", "product_id",
).agg(
    f.sum('quantity').alias('product_qty'),
    f.sum('sales_amount_net').alias('total_product_val'),
    f.sum('total_profit').alias('total_profit_product_val')
).orderBy("_id", "product_category", "product_id", "_observ_end_dt")

## Create features

In [72]:
window_client_product = Window.partitionBy(['_id', 'product_id']).orderBy(f.col("_observ_end_dt"))
window_client = Window.partitionBy(['_id']).orderBy(f.col("_observ_end_dt"))

date_range = prm_spine.select('_observ_end_dt').distinct().orderBy('_observ_end_dt')
min_date_per_client = transactions.groupby('_id', 'product_id').agg(f.min('_observ_end_dt').alias('first_trx_dt'))

max_date = date_range.agg({"_observ_end_dt": "max"}).collect()[0]["max(_observ_end_dt)"]

new_month = date_range.filter(
    date_range._observ_end_dt==max_date
).withColumn(
    '_observ_end_dt',
    f.add_months(f.col('_observ_end_dt'), 3)
)

new_date = new_month.agg({"_observ_end_dt": "max"}).collect()[0]["max(_observ_end_dt)"]


date_range = date_range.union(new_month).orderBy('_observ_end_dt')

In [74]:
client_sku = transactions.select(['_id', 'product_id',]).distinct()
client_sku_first_dt = client_sku.join(min_date_per_client, on=['_id', 'product_id'], how='inner')

sku_date_crossjoin = client_sku_first_dt.crossJoin(
    f.broadcast(date_range)
).filter(
    f.col('_observ_end_dt') >= f.col('first_trx_dt')
).drop('first_trx_dt')

transactions_after_fst_trx = transactions.join(
    sku_date_crossjoin,
    on=['_id', 'product_id', '_observ_end_dt'],
    how='right'
)

In [ ]:
working_ids = [
    # "00000540-7986-4B85-A946-E1A9D913344F__F40BF538-A9A6-49E3-9623-9301AE9FD953",
    "00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321-C6D1-4864-91B7-F00EA37C30DB",
    "00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477-4270-4547-B4CB-5706EA4479A0", ## 
    "D7185FBD-201B-49A3-B8F7-67A48705CF4A__741EBA32-9096-4557-B3AF-216AFB616384",
    "51392616-9B76-4435-8C0E-43C0139BCA1F__7272896E-F429-47A5-88F8-1195CE25FB9D",
    "AF8F3733-EE8E-493C-AD23-2F15BF80CF16__151779EE-F4CE-4F5B-9430-393D04006AAE",
    "FAB53EBC-173F-42EA-BC1F-BE6A860E5C3C__8DEE90FD-3BD0-40B4-9373-9E019AFE6BBC",
]

In [84]:
transactions_per_client = transactions_after_fst_trx.groupBy(
    ['_id', '_observ_end_dt']
).agg(
    f.sum('total_product_val').alias('total_client_net_sales'),
    f.sum('total_profit_product_val').alias('total_client_profit_sales')
).withColumn(
    'placed_order', 
    f.when(f.col('total_client_net_sales')>0,1).otherwise(0)
).withColumn(
    'total_client_orders_last_quarters',
    f.sum('placed_order').over(window_client.rowsBetween(-n_quarter_nptb_pull, -1))
).withColumn(
    'total_client_net_sales_last_quarters',
    f.sum('total_client_net_sales').over(window_client.rowsBetween(-n_quarter_nptb_pull, -1))
).withColumn(
    'total_client_profit_sales_last_quarters',
    f.sum('total_client_profit_sales').over(window_client.rowsBetween(-n_quarter_nptb_pull, -1))
).drop('placed_order')

In [85]:
transactions_per_client.filter(
    f.col("_id") == working_ids[1]
).orderBy(
    "_id", "_observ_end_dt"
).show(50, truncate=False)

+--------------------------------------------------------------------------+--------------+----------------------+-------------------------+---------------------------------+------------------------------------+---------------------------------------+
|_id                                                                       |_observ_end_dt|total_client_net_sales|total_client_profit_sales|total_client_orders_last_quarters|total_client_net_sales_last_quarters|total_client_profit_sales_last_quarters|
+--------------------------------------------------------------------------+--------------+----------------------+-------------------------+---------------------------------+------------------------------------+---------------------------------------+
|00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477-4270-4547-B4CB-5706EA4479A0|2023-01-01    |173.0434055328369     |133.9241008758545        |NULL                             |NULL                                |NULL                             

In [88]:
ftr_transactions = transactions_after_fst_trx.join(
    transactions_per_client,
    on=['_id', '_observ_end_dt'],
    how='left'
).withColumn(
    'last_transaction_quarter',
    f.when(
        f.col('total_product_val').isNotNull(),
        f.col('_observ_end_dt')
    ).otherwise(None)
).withColumn(
    'last_transaction_quarter',
    f.last('last_transaction_quarter', ignorenulls=True).over(window_client_product.rowsBetween(Window.unboundedPreceding, -1))
).withColumn(
    'total_product_val_last_order',
    f.last('total_product_val', ignorenulls=True).over(window_client_product.rowsBetween(Window.unboundedPreceding, -1))
).withColumn(
    'total_profit_product_val_last_order',
    f.last('total_profit_product_val', ignorenulls=True).over(window_client_product.rowsBetween(Window.unboundedPreceding, -1))
).withColumn(
    'units_last_order',
    f.last('product_qty', ignorenulls=True).over(window_client_product.rowsBetween(Window.unboundedPreceding, -1))
).withColumn(
    'quarters_from_last_order', #months passed since last order
    f.months_between(f.col('_observ_end_dt'),f.col('last_transaction_quarter')) / 3
).withColumn(
    'quarters_between_order', #only exists for months where order was placed
    f.when(
        f.col('total_product_val').isNotNull(),
        f.months_between(f.col('_observ_end_dt'), f.col('last_transaction_quarter')) / 3
    ).otherwise(None)
).withColumn(
    'avg_units_per_order_last_quarters',
    f.mean('product_qty').over(window_client_product.rowsBetween(-n_quarter_nptb_pull, -1))
).withColumn(
    'avg_quarters_between_orders', 
    f.mean('quarters_between_order').over(window_client_product.rowsBetween(-n_quarter_nptb_pull, -1))
).withColumn(
    'n_orders_sku_last_quarters',
    f.count('total_product_val').over(window_client_product.rowsBetween(-n_quarter_nptb_pull, -1))
).withColumn(
    'net_sales_sku_last_quarters', 
    f.sum('total_product_val').over(window_client_product.rowsBetween(-n_quarter_nptb_pull, -1))
).withColumn(
    'profit_sales_sku_last_quarters',
    f.sum('total_profit_product_val').over(window_client_product.rowsBetween(-n_quarter_nptb_pull, -1))
).withColumn(
    'sales_contribution_net',
    f.col('net_sales_sku_last_quarters') / f.col('total_client_net_sales_last_quarters')
).withColumn(
    'sales_contribution_catalog',
    f.col('profit_sales_sku_last_quarters') / f.col('total_client_profit_sales_last_quarters')
).withColumn(
    'repetition_rate',
    f.col('n_orders_sku_last_quarters') / f.col('total_client_orders_last_quarters')
).withColumn(
    'days_expected_breakage',
    f.col('units_last_order') / (f.col('avg_units_per_order_last_quarters') / f.col('avg_quarters_between_orders'))
).withColumn(
    'breakage_rate',
    f.col('quarters_from_last_order') / f.col('days_expected_breakage')
).withColumn(
    'adjusted_breakage_rate',
    f.when(
        f.col('breakage_rate') >= 1,
        f.col('breakage_rate')
    ).otherwise(0.1)
).withColumn(
    'prioritization_index',
    (f.col('quarters_from_last_order') / f.col('avg_quarters_between_orders')) * f.col('sales_contribution_catalog') * f.col('repetition_rate'))

In [89]:
ftr_transactions.show()

+--------------------+--------------+--------------------+--------------------+--------------------+----------------+-----------+------------------+------------------------+----------------------+-------------------------+---------------------------------+------------------------------------+---------------------------------------+------------------------+----------------------------+-----------------------------------+----------------+------------------------+----------------------+---------------------------------+---------------------------+--------------------------+---------------------------+------------------------------+----------------------+--------------------------+------------------+----------------------+-------------+----------------------+--------------------+
|                 _id|_observ_end_dt|          product_id|         customer_id| customer_vehicle_id|product_category|product_qty| total_product_val|total_profit_product_val|total_client_net_sales|total_client_prof

In [90]:
ftr_transactions.filter(
    f.col("_id") == working_ids[1]
).orderBy(
    "_id", "_observ_end_dt"
).show(50, truncate=False)

+--------------------------------------------------------------------------+--------------+----------------------------------------------------------------------+------------------------------------+------------------------------------+----------------+-----------+------------------+------------------------+----------------------+-------------------------+---------------------------------+------------------------------------+---------------------------------------+------------------------+----------------------------+-----------------------------------+----------------+------------------------+----------------------+---------------------------------+---------------------------+--------------------------+---------------------------+------------------------------+----------------------+--------------------------+---------------+----------------------+-------------+----------------------+--------------------+
|_id                                                                       |_obs

In [92]:
window_prioritization = Window.partitionBy(['_id', '_observ_end_dt']).orderBy(f.col("prioritization_index").desc())

ftr_transactions_filtered = ftr_transactions.filter(
    f.col("prioritization_index").isNotNull()
).withColumn(
    'ranking', 
    f.row_number().over(window_prioritization)
).filter(
    f.col("_observ_end_dt") >= new_date
).filter(
    f.col("total_client_orders_last_quarters") >= min_number_of_trx_for_recommendations
).filter(
    f.col("ranking") <= n_recommendations_pull
).withColumnRenamed(
    'product_id',
    'product_id_pull'
)

In [93]:
ftr_transactions_filtered.show(truncate=False)

+--------------------------------------------------------------------------+--------------+--------------------------------------------------------------------------------------------------------------------+-----------+-------------------+----------------+-----------+-----------------+------------------------+----------------------+-------------------------+---------------------------------+------------------------------------+---------------------------------------+------------------------+----------------------------+-----------------------------------+----------------+------------------------+----------------------+---------------------------------+---------------------------+--------------------------+---------------------------+------------------------------+----------------------+--------------------------+------------------+----------------------+------------------+----------------------+--------------------+-------+
|_id                                                         

In [96]:
out = ftr_transactions_filtered.orderBy(
    ['_id', 'product_id', '_observ_end_dt']
).select(
    '_id',
    'product_id_pull',
    'ranking'
)

In [97]:
out.show()


+--------------------+--------------------+-------+
|                 _id|     product_id_pull|ranking|
+--------------------+--------------------+-------+
|00002C41-103F-448...|TYRE SALES 17 INC...|      1|
|00007206-C3D7-423...|AIR FILTER SERVIC...|      2|
|00007206-C3D7-423...|ENGINE OIL CHANGE...|      1|
|00007206-C3D7-423...|MIGHTY OIL FILTER...|      3|
|00007206-C3D7-423...|SERVICE CHARGES 1...|      4|
|000083D3-935E-4E8...|AIR FILTER SERVIC...|      3|
|000083D3-935E-4E8...|ENGINE OIL CHANGE...|      1|
|000083D3-935E-4E8...|ENGINE OIL CHANGE...|      4|
|000083D3-935E-4E8...|OIL FILTER SERVIC...|      2|
|000083D3-935E-4E8...|TOP UP SERVICE | ...|      5|
|00009C4B-0D0D-479...|ENGINE OIL CHANGE...|      1|
|00009C4B-0D0D-479...|EXTRA | OIL | PET...|      3|
|00009C4B-0D0D-479...|OEM OIL FILTER SE...|      2|
|00009C4B-0D0D-479...|OIL FILTER SERVIC...|      4|
|00009C4B-0D0D-479...|SERVICE CHARGES 1...|      5|
|00022BBA-25EC-4F9...|ENGINE OIL CHANGE...|      1|
|00022BBA-25

# Push

## Prepare Data

In [111]:
push_transactions = int_transactions.filter(
    ~f.col('product_id').isin(products_to_drop_push)
)

initial_date_n_quarters_before = pd.to_datetime(last_date) - relativedelta(months=n_quarter_nptb_push * 3)

push_transactions_grouped = push_transactions.filter(
    f.col('_observ_end_dt') >= initial_date_n_quarters_before
).groupBy(
    ['_id', 'product_id']
).agg(
    f.countDistinct('_observ_end_dt').alias('orders')
).filter(
    f.col('orders') > minimum_orders_to_be_considered
)

## Create Sparse Matrix

In [112]:
push_transactions_grouped.show(truncate=False)

+--------------------------------------------------------------------------+--------------------------------------------------------------------------------+------+
|_id                                                                       |product_id                                                                      |orders|
+--------------------------------------------------------------------------+--------------------------------------------------------------------------------+------+
|F7946ACB-C79A-42CF-8867-BDBCF8E3313F__34372926-971C-4848-89B3-E93D9C89FDCE|ENGINE OIL CHANGE | OIL SYNTHETIC | CASTROL 10W40SYN OIL                        |5     |
|1C47FFB6-A79F-408D-9A77-8D5C7E6A17FD__7B4B979F-3BE7-4281-B2D4-2A1309578A88|ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN A-1 SUPER SYNTHETIC 5W-30 (6X4 LTR)|3     |
|D0B32CDC-5020-4077-AE4E-5D7906E983BA__527EE273-4AE9-4CA4-9B8F-E3823EFE74E5|ENGINE OIL CHANGE | OIL | FLEETMASTER LD API CH4 SAE 15W40                      |7     |
|4975BA51-

In [113]:
push_transactions_grouped.count()

1307992

In [122]:
push_transactions_grouped = push_transactions_grouped.withColumn(
    "user",
    f.hash("_id")
).withColumn(
    "item",
    f.hash("product_id")
)

In [118]:
from pyspark import ml

In [127]:
als_model = ml.recommendation.ALS(maxIter=40, rank=3, alpha=0.17, regParam=0.08, userCol="user", itemCol="item", ratingCol="orders")

als = als_model.fit(push_transactions_grouped)

In [130]:
push_transactions_grouped.filter(
    f.col("product_id").contains("SERVICE")
).orderBy("orders").select("product_id").distinct().show(truncate=False)

+--------------------------------------------------------------------------------------------------------------+
|product_id                                                                                                    |
+--------------------------------------------------------------------------------------------------------------+
|OEM OIL FILTER SERVICE | NISSAN-OIL FILTERS | OIL FILTER                                                      |
|MULTIPOINT INSPECTION | SERVICE | SERVICE                                                                     |
|OEM OIL FILTER SERVICE | GEELY-OIL FILTERS | OIL FILTER                                                       |
|AIR FILTER SERVICES | NISSAN-AIR FILTERS | AIR CLEANER ELEMENT                                                |
|OIL FILTER SERVICE | HYUNDAI-OIL FILTERS | HYUNDAI --- ENGINE OIL FILTER                                      |
|TOP UP SERVICE | GEAR OIL | PETROMIN GEAR OIL EP 140                                           

In [128]:
als.transform(push_transactions_grouped.select("_id", "user", "product_id", "item").filter("user == 874363871")).show(truncate=False)

+--------------------------------------------------------------------------+---------+--------------------------------------------------------------------------------+-----------+----------+
|_id                                                                       |user     |product_id                                                                      |item       |prediction|
+--------------------------------------------------------------------------+---------+--------------------------------------------------------------------------------+-----------+----------+
|1C47FFB6-A79F-408D-9A77-8D5C7E6A17FD__7B4B979F-3BE7-4281-B2D4-2A1309578A88|874363871|ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN A-1 SUPER SYNTHETIC 5W-30 (6X4 LTR)|-517231769 |2.9886227 |
|1C47FFB6-A79F-408D-9A77-8D5C7E6A17FD__7B4B979F-3BE7-4281-B2D4-2A1309578A88|874363871|MIGHTY OIL FILTER SERVICE | OIL FILTERS | OIL FILTER                            |-1895223630|3.8800359 |
+--------------------------------------------

In [ ]:
customer_index = {
    customer: index
    for index, customer in enumerate(push_transactions_grouped["_id"].unique())
}

item_index = {
    item: index for index, item in enumerate(push_transactions_grouped["product_id"].unique())
}

row_indices = [
    customer_index[customer] 
    for customer in push_transactions_grouped["_id"]
]
col_indices = [
    item_index[item]
    for item in push_transactions_grouped["product_id"]
]

index_customer = {
    index: customer
    for customer, index in customer_index.items()
}
index_item = {
    index: item 
    for item, index in item_index.items()
}

sparse_matrix = sp.coo_matrix(
    (list(push_transactions_grouped.orders.values), (row_indices, col_indices))
)
train_matrix = sparse_matrix.tocsr()

TypeError: 'Column' object is not callable

## Get Recommendations

In [ ]:
model = AlternatingLeastSquares(**als_params)
model.fit(train_matrix)

unique = set(row_indices)

total = len(unique)

rank_list = [i for i in range(1, recommendations_per_user + 1)]
usuarios = []
categorias = []
ranking = []
propensity = []

for user in tqdm.tqdm(unique):
    ids, scores = model.recommend(
        user,
        train_matrix[user],
        N=recommendations_per_user,
        filter_already_liked_items=True,  ## Recomienda solo prductos nuevos
    )

    item_names = [index_item[index] for index in ids]
    usuarios += [index_customer[user]] * recommendations_per_user
    categorias += item_names
    ranking += rank_list
    propensity += [i for i in scores]

recomendaciones_por_usuario = pd.DataFrame(
    {
        "_id": usuarios,
        "product_id_push": categorias,
        "ranking": ranking,
    }
)

# Results from pipeline

In [19]:
pull_df = pd.read_parquet("../data/05_model/next_product_to_buy/pull/nptb_pull_transaction_features")
push_df = pd.read_parquet("../data/05_model/next_product_to_buy/push/nptb_push_recommendations")

In [20]:
pull_df.head()

,_id,product_id_pull,ranking
0,0001DUD__966543715767,AIR FILTER SERVICES | TOYOTA-AIR FILTERS | TOY...,8
1,0001DUD__966543715767,CABIN AIR FILTER SERVICES | TOYOTA-CABIN AIR F...,3
2,0001DUD__966543715767,ENGINE FLUSH | ENGINE FLUSH | PETROMIN ENGINE ...,4
3,0001DUD__966543715767,ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN A...,1
4,0001DUD__966543715767,ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN S...,5


In [21]:
push_df.head()

,_id,product_id_push,ranking
0,7616AAB__966538316318,ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN A...,1
1,7616AAB__966538316318,OEM OIL FILTER SERVICE | TOYOTA-OIL FILTERS | ...,2
2,7616AAB__966538316318,OEM OIL FILTER SERVICE | HYUNDAI-OIL FILTERS |...,3
3,7616AAB__966538316318,TOP UP SERVICE | BRAKE FLUID | PETROMIN SUPER ...,4
4,7616AAB__966538316318,ENGINE OIL CHANGE | OIL | PETROMIN SUPER 10W30 1L,5


In [22]:
pull_df["_id"].unique().shape, push_df["_id"].unique().shape, 

((253858,), (561627,))

In [23]:
pull_df["product_id_pull"].unique().shape, push_df["product_id_push"].unique().shape, 

((713,), (60,))

In [27]:
pull_df[pull_df["_id"] == "0001DUD__966543715767"]

,_id,product_id_pull,ranking
0,0001DUD__966543715767,AIR FILTER SERVICES | TOYOTA-AIR FILTERS | TOY...,8
1,0001DUD__966543715767,CABIN AIR FILTER SERVICES | TOYOTA-CABIN AIR F...,3
2,0001DUD__966543715767,ENGINE FLUSH | ENGINE FLUSH | PETROMIN ENGINE ...,4
3,0001DUD__966543715767,ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN A...,1
4,0001DUD__966543715767,ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN S...,5
5,0001DUD__966543715767,FUEL INJECTION SERVICES | FUEL INJECTION | FUE...,2
6,0001DUD__966543715767,OEM OIL FILTER SERVICE | TOYOTA-OIL FILTERS | ...,7
7,0001DUD__966543715767,POWER STEERING SERVICES | ADDITIVES | POWER ST...,6


In [26]:
push_df["product_id_push"].value_counts()[:50]

product_id_push
ENGINE OIL CHANGE | OIL SYNTHETIC | PETROMIN A-1 SUPER SYNTHETIC 5W-30 (6X4 LTR)                354362
LUBRICATE LOCKS, HINGES AND LATCHES | SERVICE | SERVICE                                         348359
ENGINE OIL CHANGE | OIL | FLEETMASTER LD API CH4 SAE 15W40                                      291802
OEM OIL FILTER SERVICE | TOYOTA-OIL FILTERS | TOYOTA-OIL FILTERS                                274636
MULTIPOINT INSPECTION | SERVICE | SERVICE                                                       268453
MIGHTY OIL FILTER SERVICE | OIL FILTERS | OIL FILTER                                            254328
ENGINE OIL CHANGE | OIL | PETROMIN SUPER ULTRA 7 20W50 SL (6X4 LTR) CTN                         244144
GASKET  | TOYOTA-OIL FILTERS | TOYOTA-GASKET-DRAIN PLUG                                         239315
EXTRA | OIL SYNTHETIC | PETROMIN SUPER SYNTHETIC 5W30 - 1L                                      223975
TOP UP SERVICE | BRAKE FLUID | PETROMIN SUPER BRAKE FLUID